# hakuhodo24 修正版
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryouy/hakuhodo24-repository/blob/main/hakuhodo24.ipynb)

Google Colabで上から順に実行してください。APIキーは Colab のシークレット `api_key_sc` から取得します。入力ファイル名や保存先は設定セルで変更できます。

In [ ]:
!pip -q install -U openai python-pptx pdf2image Pillow requests matplotlib
!apt-get -qq update && apt-get -qq install -y poppler-utils

In [ ]:
import os, re, json, base64, random, time
from pathlib import Path
from io import BytesIO
from PIL import Image
from pdf2image import convert_from_path
from pptx import Presentation
from openai import OpenAI
from google.colab import drive, userdata
from IPython.display import display

drive.mount('/content/drive')
api_key = userdata.get('api_key_sc')
if not api_key:
    raise RuntimeError('Colabのシークレット api_key_sc が見つかりません。ノートブックからのアクセスも有効にしてください。')
client = OpenAI(api_key=api_key)

SOURCE_DIR = Path('/content/drive/MyDrive/ブランチズム紹介資料/Splannt')
PDF_NAME = '[Splannt]サービス紹介スライド.pdf'
PPTX_NAME = '[Splannt]サービス紹介スライド.pptx'
WORK_DIR = SOURCE_DIR / 'hakuhodo24_output'
SLIDES_DIR = WORK_DIR / 'slides'
GENERATED_DIR = WORK_DIR / 'generated_images'
for d in (WORK_DIR, SLIDES_DIR, GENERATED_DIR): d.mkdir(parents=True, exist_ok=True)
TEXT_MODEL = 'gpt-4.1-mini'
IMAGE_MODEL = 'gpt-image-1'
print('出力先:', WORK_DIR)

In [ ]:
def require_file(path):
    if not path.exists(): raise FileNotFoundError(f'入力ファイルが見つかりません: {path}')

def text_chat(prompt, max_output_tokens=600):
    r = client.responses.create(model=TEXT_MODEL, input=prompt, max_output_tokens=max_output_tokens)
    return r.output_text.strip()

def image_data_url(path):
    mime = 'image/png' if path.suffix.lower() == '.png' else 'image/jpeg'
    data = base64.b64encode(path.read_bytes()).decode('ascii')
    return f'data:{mime};base64,{data}'

def vision_chat(question, image_path, max_output_tokens=400):
    r = client.responses.create(model=TEXT_MODEL, input=[{'role':'user','content':[
        {'type':'input_text','text':question},
        {'type':'input_image','image_url':image_data_url(Path(image_path))}
    ]}], max_output_tokens=max_output_tokens)
    return r.output_text.strip()

def generate_image(prompt, save_path):
    r = client.images.generate(model=IMAGE_MODEL, prompt=prompt, size='1024x1024')
    item = r.data[0]
    if getattr(item, 'b64_json', None):
        raw = base64.b64decode(item.b64_json)
    else:
        import requests
        raw = requests.get(item.url, timeout=60).content
    Path(save_path).write_bytes(raw)
    return Path(save_path)

def retry(fn, *args, attempts=3, **kwargs):
    for i in range(attempts):
        try: return fn(*args, **kwargs)
        except Exception:
            if i == attempts - 1: raise
            time.sleep(2 ** i)

In [ ]:
pdf_path, pptx_path = SOURCE_DIR / PDF_NAME, SOURCE_DIR / PPTX_NAME
require_file(pdf_path)
require_file(pptx_path)

images = convert_from_path(str(pdf_path), dpi=150)
for i, image in enumerate(images, 1):
    image.save(SLIDES_DIR / f'slide_{i:03d}.png', 'PNG')

prs = Presentation(str(pptx_path))
for i, slide in enumerate(prs.slides, 1):
    parts = [shape.text.strip() for shape in slide.shapes if hasattr(shape, 'text') and shape.text.strip()]
    (SLIDES_DIR / f'slide_{i:03d}.txt').write_text('\n'.join(parts), encoding='utf-8')

if len(images) != len(prs.slides):
    print(f'注意: PDFは{len(images)}ページ、PPTXは{len(prs.slides)}枚です。共通するページを処理します。')
slide_count = min(len(images), len(prs.slides))
print('処理対象:', slide_count, 'ページ')

In [ ]:
page_summaries = []
for i in range(1, slide_count + 1):
    img = SLIDES_DIR / f'slide_{i:03d}.png'
    txt = (SLIDES_DIR / f'slide_{i:03d}.txt').read_text(encoding='utf-8')
    visual = retry(vision_chat, 'このプレゼンページを日本語200文字以内で具体的に説明してください。', img)
    prompt = f'''プレゼンの第{i}ページを日本語で簡潔に要約してください。サービス内容、理念、重要点を優先し、特殊な装飾文字は使わないでください。
画像の説明:
{visual}
抽出テキスト:
{txt}'''
    summary = retry(text_chat, prompt, 500)
    (SLIDES_DIR / f'slide_{i:03d}_analysis.txt').write_text(summary, encoding='utf-8')
    page_summaries.append(f'ページ{i}: {summary}')
    print(f'{i}/{slide_count} 完了')
presentation_summary = '\n'.join(page_summaries)
(WORK_DIR / 'presentation_summary.txt').write_text(presentation_summary, encoding='utf-8')
print(presentation_summary)

In [ ]:
def discuss(phase_instruction, context, turns=3):
    personas = retry(text_chat, f'''次のテーマを議論する、互いに異なる大学生3名のペルソナを作ってください。各人は名前、背景、価値観、口調を持ちます。
テーマ: {context[:5000]}''', 800)
    log = []
    for turn in range(turns):
        prompt = f'''3名の大学生による議論を進めます。
ペルソナ:
{personas}
指示: {phase_instruction}
前提:
{context[:8000]}
これまでの発言:
{json.dumps(log, ensure_ascii=False)}
重複を避け、次の1名の発言を100字以内で出力してください。名前も付けてください。'''
        log.append(retry(text_chat, prompt, 250))
    return retry(text_chat, f'''以下の議論を200字以内で要約してください。企業名や元サービス名には触れず、新規ITサービス開発につながる内容にしてください。
{log}''', 400)

phases = [
 '資料を踏まえ、学生が抱える悩みを当事者意識を持って推測する',
 '悩みから生じる具体的かつ深刻な問題を挙げる',
 '問題から派生する課題を幅広く挙げる',
 '課題の共通点を俯瞰して特定する',
 'ITを使った解決の方向性を具体化する',
 '留学生にも有効な新規ITサービス案を具体化する'
]
context = presentation_summary
phase_results = []
for i, instruction in enumerate(phases, 1):
    context = discuss(instruction, context, turns=3)
    phase_results.append(context)
    print(f'フェーズ{i}: {context}\n')
service_conclusion = phase_results[-1]
(WORK_DIR / 'discussion_results.txt').write_text('\n\n'.join(phase_results), encoding='utf-8')

In [ ]:
idea_prompt = f'''次の結論から留学生向けITサービス案をJSONだけで作成してください。
結論: {service_conclusion}
形式: {{"service_name":"短い名前","sub_theme":"短い副題","problems":["問題1","問題2","問題3"],"solutions":["解決策1","解決策2"],"change":"導入後の変化を示す1文"}}'''
raw_idea = retry(text_chat, idea_prompt, 700)
raw_idea = re.sub(r'^```(?:json)?|```$', '', raw_idea.strip(), flags=re.MULTILINE).strip()
idea = json.loads(raw_idea)

prompts = [
 f'留学生が直面する課題を象徴する、文字なしの洗練されたプレゼン用イラスト。{idea["problems"]}',
 f'{idea["service_name"]}というITサービスを象徴する、文字なしの洗練されたプレゼン用イラスト。{idea["solutions"]}',
 f'留学生が支援を得て前向きに変化する様子。文字なしの洗練されたプレゼン用イラスト。{idea["change"]}'
]
generated = []
for i, prompt in enumerate(prompts, 1):
    path = retry(generate_image, prompt, GENERATED_DIR / f'image_{i}.png')
    generated.append(path)
    display(Image.open(path))

def bullets(items): return '\n'.join(f'- {x}' for x in items)
marp = f'''---
marp: true
theme: uncover
paginate: true
---
# {idea['service_name']}
## {idea['sub_theme']}

![bg left:40%]({generated[1].as_posix()})
---
# 解決する問題
{bullets(idea['problems'])}

![bg right:40%]({generated[0].as_posix()})
---
# 提供する解決策
{bullets(idea['solutions'])}

![bg left:40%]({generated[1].as_posix()})
---
# もたらす変化
{idea['change']}

![bg right:50%]({generated[2].as_posix()})
'''
marp_path = WORK_DIR / 'service_pitch.md'
marp_path.write_text(marp, encoding='utf-8')
(WORK_DIR / 'service_idea.json').write_text(json.dumps(idea, ensure_ascii=False, indent=2), encoding='utf-8')
print(marp)
print('完了。成果物:', WORK_DIR)